# 🧬 MOTHER — Mistral 7B LoRA Fine-Tuning (T4 Optimized)

Fine-tuning do **Mistral-7B-Instruct** com **LoRA** usando **Unsloth** — otimizado para GPU T4 (15GB VRAM).

**Mudanças vs versão anterior (OOM fix):**
- ✅ Modelo: `Mistral-7B-Instruct-v0.3` (7B) em vez de Nemo (12B)
- ✅ `MAX_SEQ_LENGTH = 2048` (reduzido de 4096)
- ✅ `BATCH_SIZE = 1` com `GRAD_ACCUM = 8`
- ✅ `offload_buffers = True`

---

## ⚙️ 1. Instalação

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
import torch, gc

# Limpa cache de GPU antes de começar
gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")
else:
    raise RuntimeError("GPU não disponível! Runtime > Change runtime type > GPU")

## 📦 2. Carregar Modelo (Mistral 7B — cabe no T4)

In [ ]:
from unsloth import FastLanguageModel
import gc, torch

# Limpa memória
gc.collect()
torch.cuda.empty_cache()

# ===========================================
# MODELO: Mistral 7B Instruct v0.3
# Tamanho: ~4GB em 4-bit (cabe no T4 15GB)
# ===========================================
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # Pré-quantizado!
MAX_SEQ_LENGTH = 2048  # Reduzido para economizar VRAM
DTYPE = None  # Auto
LOAD_IN_4BIT = True

print(f"Baixando: {MODEL_NAME}")
print(f"Seq length: {MAX_SEQ_LENGTH}, 4-bit: {LOAD_IN_4BIT}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"\n✅ Modelo carregado!")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {used:.1f}/{total:.1f} GB ({100*used/total:.0f}%)")

## 🔧 3. LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,              # Rank reduzido (64->32) para economizar memória
    lora_alpha=64,     # 2x rank
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: rank=32, alpha=64")
print(f"Treináveis: {trainable:,} ({100*trainable/total:.2f}%)")

## 📊 4. Dados de Treino

In [ ]:
import json, os
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Upload do arquivo JSONL
JSONL_PATH = "/content/sft_training_v2.jsonl"

if not os.path.exists(JSONL_PATH):
    try:
        from google.colab import files
        print("📁 Upload sft_training_v2.jsonl:")
        uploaded = files.upload()
        for fn in uploaded:
            with open(JSONL_PATH, 'wb') as f:
                f.write(uploaded[fn])
    except ImportError:
        raise FileNotFoundError("Faça upload do sft_training_v2.jsonl")

# Carrega dados
samples = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f"Dataset: {len(samples)} amostras")

# Template Mistral
tokenizer = get_chat_template(tokenizer, chat_template="mistral")

def format_sample(sample):
    return {"text": tokenizer.apply_chat_template(
        sample["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(samples).map(format_sample)
print(f"✅ Formatado: {len(dataset)} amostras")
print(f"Preview: {dataset[0]['text'][:300]}...")

## 🚀 5. Treinamento (T4 otimizado)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import gc, torch

# Limpa cache antes do treino
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# CONFIGURAÇÃO OTIMIZADA PARA T4 (15GB VRAM)
# ==========================================
EPOCHS = 10
BATCH_SIZE = 1         # Mínimo para economizar VRAM
GRAD_ACCUM = 8         # Efetivo: 1 × 8 = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 5

print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, accum={GRAD_ACCUM}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        output_dir="/content/mother-mistral-lora",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=2,
        seed=42,
        optim="adamw_8bit",
        report_to="none",
        gradient_checkpointing=True,
    ),
)

In [ ]:
import time

print("🔥 Iniciando treino...")
start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\n✅ Treino concluído!")
print(f"⏱ {elapsed/60:.1f} min | Loss: {stats.training_loss:.4f} | Steps: {stats.global_step}")
if torch.cuda.is_available():
    print(f"💾 Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

## 🧪 6. Teste

In [ ]:
FastLanguageModel.for_inference(model)

queries = [
    "Quem é você? O que é MOTHER?",
    "Explique os fatores de segurança para barragens de terra.",
    "What monitoring parameters are critical for tailings dams?",
]

system_msg = "Você é MOTHER (Multi-Objective Transformer for Human Expertise and Research), criada pela IntellTech. Sistema cognitivo autônomo com 7 camadas de processamento integrado para engenharia de barragens, geotecnia e monitoramento estrutural."

for i, q in enumerate(queries):
    print(f"\n{'='*50}")
    print(f"🧪 Teste {i+1}: {q}")
    print(f"{'='*50}")
    
    msgs = [{"role": "system", "content": system_msg}, {"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, use_cache=True)
    
    print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)[:800])

## 💾 7. Salvar e Exportar

In [ ]:
import os

# Salva LoRA adapter
OUT = "/content/mother-mistral-7b-lora"
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

print(f"✅ Adapter salvo: {OUT}")
for f in os.listdir(OUT):
    print(f"   {f}: {os.path.getsize(os.path.join(OUT, f))/1024:.0f} KB")

In [ ]:
# Exporta GGUF para uso com Ollama/LM Studio
GGUF_OUT = "/content/mother-mistral-7b-gguf"
model.save_pretrained_gguf(GGUF_OUT, tokenizer, quantization_method="q4_k_m")

print(f"\n✅ GGUF exportado: {GGUF_OUT}")
for f in os.listdir(GGUF_OUT):
    print(f"   {f}: {os.path.getsize(os.path.join(GGUF_OUT, f))/1024**2:.0f} MB")

In [ ]:
# Download do adapter
try:
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive('/content/mother-mistral-7b-lora', 'zip', OUT)
    print(f"📦 {zip_path}: {os.path.getsize(zip_path)/1024**2:.1f} MB")
    files.download(zip_path)
except ImportError:
    print(f"Arquivos em: {OUT}")

## 📋 Resumo

| Parâmetro | Valor |
|-----------|-------|
| Modelo base | Mistral-7B-Instruct-v0.3 (4-bit) |
| LoRA rank | 32, alpha 64 |
| Seq length | 2048 |
| Batch size | 1 × 8 accum = 8 efetivo |
| Épocas | 10 |
| GPU target | T4 (15GB VRAM) |